# Notebook 08B - Pump Models Training (RUL, Failure Probability, Failure Mode)


In [2]:
import pandas as pd
import numpy as np
import joblib
!pip install xgboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score
from xgboost import XGBRegressor, XGBClassifier


ModuleNotFoundError: No module named 'xgboost'

In [ ]:
telemetry = pd.read_csv('pump_telemetry.csv')
labels = pd.read_csv('pump_labels.csv')
df = telemetry.merge(labels, on=['timestamp','asset_id'])


In [ ]:
FEATURES = ['bearing_temp','motor_temp','vibration','flow_rate','suction_pressure','discharge_pressure','motor_current','lubrication_level','seal_leakage_rate']
X = df[FEATURES]


## Model 1 - RUL Prediction


In [ ]:
y_rul = df['rul_days']
X_train,X_test,y_train,y_test = train_test_split(X,y_rul,test_size=0.2,random_state=42)
rul_model = XGBRegressor(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
rul_model.fit(X_train,y_train)
pred = rul_model.predict(X_test)
print('MAE:', mean_absolute_error(y_test,pred))
print('RMSE:', np.sqrt(mean_squared_error(y_test,pred)))
print('R2:', r2_score(y_test,pred))
joblib.dump(rul_model,'pump_rul_model.pkl')


## Model 2 - Failure Probability (Next 30 Days)


In [ ]:
y_fail = df['failure_next_30_days']
X_train,X_test,y_train,y_test = train_test_split(X,y_fail,test_size=0.2,random_state=42)
failure_model = XGBClassifier(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
failure_model.fit(X_train,y_train)
pred = failure_model.predict(X_test)
print('Accuracy:', accuracy_score(y_test,pred))
joblib.dump(failure_model,'pump_failure_probability_model.pkl')


## Model 3 - Failure Mode Classification


In [ ]:
encoder = LabelEncoder()
y_mode = encoder.fit_transform(df['failure_mode'])
X_train,X_test,y_train,y_test = train_test_split(X,y_mode,test_size=0.2,random_state=42)
mode_model = XGBClassifier(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
mode_model.fit(X_train,y_train)
pred = mode_model.predict(X_test)
print('Accuracy:', accuracy_score(y_test,pred))
joblib.dump(mode_model,'pump_failure_mode_model.pkl')
joblib.dump(encoder,'failure_mode_encoder.pkl')
